CHB-MIT Scalp EEG Database 
==================================================
Dataset: https://physionet.org/content/chbmit/1.0.0/

This dataset contains EEG recordings from pediatric subjects with 
intractable seizures. Key characteristics:
- 23 subjects (chb01-chb24, no chb21)
- 844 hours of continuous EEG
- 198 seizures annotated
- 23 channels (international 10-20 system)
- 256 Hz sampling rate

For NeuroState project:
- Used for seizure detection downstream task
- Changepoints align with seizure onset/offset
- Provides pathological EEG for diverse pretraining

In [2]:
# IMPORTS (with pandas Arrow fix)
import os
import numpy as np

# Fix pandas Arrow string issue BEFORE other imports
import pandas as pd
pd.options.future.infer_string = False

import mne
from pathlib import Path
from tqdm import tqdm
import h5py
import re
import warnings
warnings.filterwarnings('ignore')

# Optional: for downloading from PhysioNet
try:
    import wfdb
    WFDB_AVAILABLE = True
except ImportError:
    WFDB_AVAILABLE = False
    print("Note: wfdb not installed. Manual download required.")

print("Imports successful!")

Imports successful!


In [3]:
# CONFIGURATION
class CHBMITConfig:
    """Configuration for CHB-MIT preprocessing."""
    
    # Paths
    DATA_DIR = Path("./data")
    RAW_DIR = DATA_DIR / "raw" / "chb-mit"
    PROCESSED_DIR = DATA_DIR / "processed"
    FIGURES_DIR = Path("./figures")
    
    # Dataset info
    DATASET_NAME = "chb-mit"
    PHYSIONET_URL = "https://physionet.org/content/chbmit/1.0.0/"
    
    # Subjects (chb21 doesn't exist in dataset)
    SUBJECTS = [f"chb{i:02d}" for i in range(1, 25) if i != 21]
    
    # EEG Parameters
    SFREQ_ORIGINAL = 256  # CHB-MIT sampling frequency
    SFREQ_TARGET = 100    # Resample to match Sleep-EDF for unified pipeline
    
    # Filtering
    L_FREQ = 0.5
    H_FREQ = 35.0
    NOTCH_FREQ = 60.0  # US data - 60Hz line noise
    
    # Standard 10-20 channels in CHB-MIT (may vary slightly per subject)
    STANDARD_CHANNELS = [
        'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
        'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
        'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
        'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
        'FZ-CZ', 'CZ-PZ',
        'P7-T7', 'T7-FT9', 'FT9-FT10', 'FT10-T8'
    ]
    
    # Minimum channels required
    MIN_CHANNELS = 18
    
    # Epoching
    EPOCH_DURATION = 30.0  # seconds - same as Sleep-EDF for consistency
    EPOCH_OVERLAP = 0.0
    
    # For seizure detection task
    SEIZURE_WINDOW = 30.0      # Window size for seizure epochs
    PRE_SEIZURE_BUFFER = 5.0   # Include 5s before seizure onset
    POST_SEIZURE_BUFFER = 5.0  # Include 5s after seizure offset
    
    # Labels
    LABEL_NORMAL = 0
    LABEL_SEIZURE = 1
    
    # Normalization
    NORMALIZATION = "zscore"
    
    # Artifact rejection
    ARTIFACT_THRESHOLD_UV = 500
    
    # Random seed
    SEED = 42

# Create directories
for dir_path in [CHBMITConfig.DATA_DIR, CHBMITConfig.RAW_DIR, 
                 CHBMITConfig.PROCESSED_DIR, CHBMITConfig.FIGURES_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

In [4]:
# DOWNLOAD CHB-MIT DATA (12 SUBJECTS)

import os
import subprocess

def download_chbmit_subjects(subjects=None, data_dir=None):
    """
    Download specific CHB-MIT subjects using wget.
    
    Parameters:
    -----------
    subjects : list
        Subject IDs to download (e.g., ['chb01', 'chb03'])
    data_dir : Path
        Directory to save data
    """
    if subjects is None:
        subjects = ["chb01", "chb03", "chb05", "chb08", "chb10"]  # Default 5 subjects
    
    if data_dir is None:
        data_dir = CHBMITConfig.RAW_DIR
    
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Downloading {len(subjects)} CHB-MIT subjects to {data_dir}")
    print(f"Subjects: {subjects}\n")
    
    base_url = "https://physionet.org/files/chbmit/1.0.0"
    
    for subj in subjects:
        subj_dir = data_dir / subj
        
        # Skip if already downloaded
        if subj_dir.exists() and len(list(subj_dir.glob("*.edf"))) > 0:
            n_files = len(list(subj_dir.glob("*.edf")))
            print(f"✓ {subj}: Already exists ({n_files} EDF files)")
            continue
        
        print(f" Downloading {subj}...")
        
        url = f"{base_url}/{subj}/"
        cmd = [
            "wget", "-r", "-np", "-nH", 
            "--cut-dirs=3",  # Remove 'physionet.org/files/chbmit/1.0.0' from path
            "-R", "index.html*",
            "-P", str(data_dir),
            "-q", "--show-progress",  # Quiet but show progress
            url
        ]
        
        try:
            subprocess.run(cmd, check=True)
            n_files = len(list((data_dir / subj).glob("*.edf")))
            print(f"  ✓ {subj}: Downloaded ({n_files} EDF files)")
        except subprocess.CalledProcessError as e:
            print(f"  ✗ {subj}: Download failed - {e}")
        except FileNotFoundError:
            # wget not available, try alternative
            print(f"  wget not found, trying alternative method...")
            os.system(f'wget -r -np -nH --cut-dirs=3 -R "index.html*" -P {data_dir} {url}')
    
    # Verify download
    print("\n" + "=" * 50)
    print("DOWNLOAD SUMMARY")
    print("=" * 50)
    
    total_files = 0
    for subj in subjects:
        subj_dir = data_dir / subj
        if subj_dir.exists():
            n_edf = len(list(subj_dir.glob("*.edf")))
            has_summary = (subj_dir / f"{subj}-summary.txt").exists()
            total_files += n_edf
            status = "✓" if n_edf > 0 and has_summary else "⚠"
            print(f"  {status} {subj}: {n_edf} EDF files, summary: {has_summary}")
        else:
            print(f"  ✗ {subj}: NOT FOUND")
    
    print(f"\nTotal EDF files: {total_files}")
    return subjects


# EXECUTE DOWNLOAD
# Check what's already downloaded
existing_subjects = []
if CHBMITConfig.RAW_DIR.exists():
    existing_subjects = [d.name for d in CHBMITConfig.RAW_DIR.iterdir() 
                         if d.is_dir() and d.name.startswith("chb")]

print(f"Already downloaded: {existing_subjects if existing_subjects else 'None'}\n")

# Download 12 subjects 
SUBJECTS_TO_DOWNLOAD = ["chb01", "chb03", "chb05", "chb08", "chb10", "chb12", "chb14", "chb15", "chb17", "chb19", "chb20", "chb22"]

downloaded_subjects = download_chbmit_subjects(
    subjects=SUBJECTS_TO_DOWNLOAD,
    data_dir=CHBMITConfig.RAW_DIR
)

Already downloaded: ['chb01', 'chb10', 'chb15', 'chb19', 'chb03', 'chb17', 'chb08', 'chb05', 'chb12', 'chb14', 'chb22', 'chb20']

Subjects: ['chb01', 'chb03', 'chb05', 'chb08', 'chb10', 'chb12', 'chb14', 'chb15', 'chb17', 'chb19', 'chb20', 'chb22']

✓ chb01: Already exists (42 EDF files)
✓ chb03: Already exists (38 EDF files)
✓ chb05: Already exists (39 EDF files)
✓ chb08: Already exists (20 EDF files)
✓ chb10: Already exists (25 EDF files)
✓ chb12: Already exists (24 EDF files)
✓ chb14: Already exists (26 EDF files)
✓ chb15: Already exists (40 EDF files)
✓ chb17: Already exists (21 EDF files)
✓ chb19: Already exists (30 EDF files)
✓ chb20: Already exists (29 EDF files)
✓ chb22: Already exists (31 EDF files)

DOWNLOAD SUMMARY
  ✓ chb01: 42 EDF files, summary: True
  ✓ chb03: 38 EDF files, summary: True
  ✓ chb05: 39 EDF files, summary: True
  ✓ chb08: 20 EDF files, summary: True
  ✓ chb10: 25 EDF files, summary: True
  ✓ chb12: 24 EDF files, summary: True
  ✓ chb14: 26 EDF files, summa

In [5]:
# SUMMARY FILE PARSER
class CHBMITSummaryParser:
    """
    Parse CHB-MIT summary files to extract seizure annotations.
    
    Each subject folder contains a SUBJECT-summary.txt file with:
    - File names
    - Recording times
    - Seizure start/end times (in seconds from file start)
    """
    
    def __init__(self, subject_dir):
        """
        Parameters:
        -----------
        subject_dir : Path
            Path to subject folder (e.g., ./data/raw/chb-mit/chb01/)
        """
        self.subject_dir = Path(subject_dir)
        self.subject_id = self.subject_dir.name
        self.summary_file = self.subject_dir / f"{self.subject_id}-summary.txt"
        
    def parse(self):
        """
        Parse the summary file.
        
        Returns:
        --------
        list : List of dicts with file info and seizure annotations
        """
        if not self.summary_file.exists():
            raise FileNotFoundError(f"Summary file not found: {self.summary_file}")
        
        with open(self.summary_file, 'r') as f:
            content = f.read()
        
        # Split by "File Name:" to get individual file entries
        file_blocks = re.split(r'File Name:', content)[1:]  # Skip header
        
        files_info = []
        
        for block in file_blocks:
            info = self._parse_file_block(block)
            if info:
                files_info.append(info)
        
        return files_info
    
    def _parse_file_block(self, block):
        """Parse a single file block from the summary."""
        lines = block.strip().split('\n')
        
        if not lines:
            return None
        
        info = {
            'filename': lines[0].strip(),
            'seizures': []
        }
        
        # Parse seizure info
        n_seizures = 0
        for line in lines:
            line = line.strip()
            
            # Number of seizures
            if 'Number of Seizures' in line:
                match = re.search(r'(\d+)', line)
                if match:
                    n_seizures = int(match.group(1))
            
            # Seizure start time
            elif 'Seizure Start Time' in line or 'Seizure.*Start Time' in line:
                match = re.search(r'(\d+)', line)
                if match:
                    start_time = int(match.group(1))
                    info['seizures'].append({'start': start_time})
            
            # Seizure end time
            elif 'Seizure End Time' in line or 'Seizure.*End Time' in line:
                match = re.search(r'(\d+)', line)
                if match and info['seizures']:
                    info['seizures'][-1]['end'] = int(match.group(1))
        
        return info

In [6]:
# PREPROCESSING CLASS
class CHBMITPreprocessor:
    """
    Preprocessing pipeline for CHB-MIT seizure dataset.
    
    Key differences from Sleep-EDF:
    1. Higher original sampling rate (256 Hz vs 100 Hz)
    2. More channels (23 vs 2)
    3. US data requires 60Hz notch filter
    4. Binary labels (seizure/normal) instead of 5-class sleep stages
    5. Variable-length seizure events vs fixed 30s epochs
    """
    
    def __init__(self, config=CHBMITConfig):
        self.config = config
        self.summary_parser = None
    
    def load_edf(self, edf_path, verbose=False):
        """
        Load a single EDF file.
        
        Parameters:
        -----------
        edf_path : str or Path
            Path to EDF file
            
        Returns:
        --------
        mne.io.Raw : Raw EEG data
        """
        raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=verbose)
        return raw
    
    def standardize_channels(self, raw):
        """
        Standardize channel names and select common channels.
        
        CHB-MIT has some variation in channel names across subjects.
        """
        # Get available channels
        ch_names = raw.ch_names
        
        # Map common variations
        rename_map = {}
        for ch in ch_names:
            # Remove dots and standardize
            std_name = ch.replace('.', '').upper()
            std_name = std_name.replace('--', '-')
            
            # Handle common patterns
            if std_name != ch:
                rename_map[ch] = std_name
        
        if rename_map:
            raw = raw.rename_channels(rename_map)
        
        # Find channels that match standard 10-20 patterns
        eeg_channels = []
        for ch in raw.ch_names:
            ch_upper = ch.upper()
            # Check if it's a bipolar EEG channel
            if '-' in ch_upper and not any(x in ch_upper for x in ['ECG', 'EKG', 'VNS', 'STI']):
                eeg_channels.append(ch)
        
        if len(eeg_channels) < self.config.MIN_CHANNELS:
            raise ValueError(
                f"Only {len(eeg_channels)} EEG channels found, "
                f"need at least {self.config.MIN_CHANNELS}"
            )
        
        # Pick only EEG channels
        raw = raw.pick_channels(eeg_channels)
        
        return raw
    
    def apply_filters(self, raw):
        """Apply bandpass and notch filters."""
        # Notch filter for 60Hz line noise (US data)
        if self.config.NOTCH_FREQ:
            raw = raw.notch_filter(
                freqs=self.config.NOTCH_FREQ,
                verbose=False
            )
        
        # Bandpass filter
        raw = raw.filter(
            l_freq=self.config.L_FREQ,
            h_freq=self.config.H_FREQ,
            fir_design='firwin',
            verbose=False
        )
        
        return raw
    
    def resample(self, raw):
        """Resample to target frequency."""
        if raw.info['sfreq'] != self.config.SFREQ_TARGET:
            raw = raw.resample(self.config.SFREQ_TARGET, verbose=False)
        return raw
    
    def create_seizure_epochs(self, raw, seizure_annotations):
        """
        Create epochs around seizure events and matched normal epochs.
        
        Parameters:
        -----------
        raw : mne.io.Raw
            Preprocessed raw data
        seizure_annotations : list
            List of dicts with 'start' and 'end' keys (in seconds)
            
        Returns:
        --------
        np.ndarray : Epoch data (n_epochs, n_channels, n_samples)
        np.ndarray : Labels (0=normal, 1=seizure)
        np.ndarray : Epoch metadata (start_time, end_time, type)
        """
        sfreq = raw.info['sfreq']
        epoch_samples = int(self.config.SEIZURE_WINDOW * sfreq)
        data = raw.get_data()
        duration = data.shape[1] / sfreq
        
        epochs_list = []
        labels_list = []
        metadata_list = []
        
        # Create seizure epochs
        seizure_times = []
        for seizure in seizure_annotations:
            start = seizure['start'] - self.config.PRE_SEIZURE_BUFFER
            end = seizure['end'] + self.config.POST_SEIZURE_BUFFER
            
            # Create overlapping windows covering the seizure
            t = max(0, start)
            while t + self.config.SEIZURE_WINDOW <= min(end, duration):
                start_sample = int(t * sfreq)
                end_sample = start_sample + epoch_samples
                
                if end_sample <= data.shape[1]:
                    epoch = data[:, start_sample:end_sample]
                    if epoch.shape[1] == epoch_samples:
                        epochs_list.append(epoch)
                        labels_list.append(self.config.LABEL_SEIZURE)
                        metadata_list.append({
                            'start': t,
                            'end': t + self.config.SEIZURE_WINDOW,
                            'type': 'seizure'
                        })
                        seizure_times.append((t, t + self.config.SEIZURE_WINDOW))
                
                t += self.config.SEIZURE_WINDOW / 2  # 50% overlap for seizures
        
        # Create normal epochs (avoiding seizure periods)
        n_seizure_epochs = len(epochs_list)
        n_normal_needed = n_seizure_epochs * 3  # 3:1 ratio normal:seizure
        
        # Find valid normal periods
        normal_epochs_added = 0
        t = 0
        while normal_epochs_added < n_normal_needed and t + self.config.SEIZURE_WINDOW <= duration:
            # Check if this window overlaps with any seizure
            is_seizure = any(
                not (t + self.config.SEIZURE_WINDOW < s_start or t > s_end)
                for s_start, s_end in seizure_times
            )
            
            if not is_seizure:
                start_sample = int(t * sfreq)
                end_sample = start_sample + epoch_samples
                
                if end_sample <= data.shape[1]:
                    epoch = data[:, start_sample:end_sample]
                    if epoch.shape[1] == epoch_samples:
                        epochs_list.append(epoch)
                        labels_list.append(self.config.LABEL_NORMAL)
                        metadata_list.append({
                            'start': t,
                            'end': t + self.config.SEIZURE_WINDOW,
                            'type': 'normal'
                        })
                        normal_epochs_added += 1
            
            t += self.config.SEIZURE_WINDOW  # No overlap for normal epochs
        
        if len(epochs_list) == 0:
            return None, None, None
        
        epochs = np.array(epochs_list)
        labels = np.array(labels_list)
        
        return epochs, labels, metadata_list
    
    def create_pretraining_epochs(self, raw, seizure_annotations=None):
        """
        Create fixed-length epochs for pretraining (ignoring seizure labels).
        
        This creates 30-second epochs across the entire recording,
        suitable for self-supervised pretraining.
        """
        sfreq = raw.info['sfreq']
        epoch_samples = int(self.config.EPOCH_DURATION * sfreq)
        data = raw.get_data()
        
        epochs_list = []
        times_list = []
        
        # Create non-overlapping epochs
        n_epochs = data.shape[1] // epoch_samples
        
        for i in range(n_epochs):
            start_sample = i * epoch_samples
            end_sample = start_sample + epoch_samples
            
            epoch = data[:, start_sample:end_sample]
            epochs_list.append(epoch)
            times_list.append(start_sample / sfreq)
        
        epochs = np.array(epochs_list)
        times = np.array(times_list)
        
        # Mark which epochs contain seizures (for analysis, not training)
        seizure_mask = np.zeros(len(epochs), dtype=bool)
        if seizure_annotations:
            for seizure in seizure_annotations:
                for i, t in enumerate(times_list):
                    if seizure['start'] <= t + self.config.EPOCH_DURATION and \
                       seizure['end'] >= t:
                        seizure_mask[i] = True
        
        return epochs, times, seizure_mask
    
    def normalize_epochs(self, epochs, method='zscore'):
        """Z-score normalization per epoch per channel."""
        normalized = np.zeros_like(epochs)
        
        for i in range(epochs.shape[0]):
            for j in range(epochs.shape[1]):
                segment = epochs[i, j, :]
                mean = np.mean(segment)
                std = np.std(segment)
                if std > 0:
                    normalized[i, j, :] = (segment - mean) / std
                else:
                    normalized[i, j, :] = segment - mean
        
        return normalized
    
    def detect_artifacts(self, epochs, threshold_uv=500):
        """Detect bad epochs based on amplitude."""
        # Convert to microvolts if needed
        data_uv = epochs * 1e6 if np.abs(epochs).max() < 1 else epochs
        ptp = np.ptp(data_uv, axis=2).max(axis=1)
        return ptp > threshold_uv
    
    def process_subject(self, subject_id, mode='seizure_detection', verbose=True):
        """
        Process all files for a single subject.
        """
        subject_dir = self.config.RAW_DIR / subject_id
        
        if not subject_dir.exists():
            raise FileNotFoundError(f"Subject directory not found: {subject_dir}")
        
        if verbose:
            print(f"\nProcessing {subject_id}...")
        
        # Parse summary file
        parser = CHBMITSummaryParser(subject_dir)
        files_info = parser.parse()
        
        all_epochs = []
        all_labels = []
        all_metadata = []
        
        for file_info in tqdm(files_info, desc=f"  {subject_id} files", disable=not verbose):
            edf_path = subject_dir / file_info['filename']
            
            if not edf_path.exists():
                continue
            
            try:
                # Load and preprocess
                raw = self.load_edf(edf_path, verbose=False)
                raw = self.standardize_channels(raw)
                raw = self.apply_filters(raw)
                raw = self.resample(raw)
                
                # Create epochs based on mode
                if mode == 'seizure_detection':
                    epochs, labels, metadata = self.create_seizure_epochs(
                        raw, file_info['seizures']
                    )
                else:  # pretraining
                    epochs, times, seizure_mask = self.create_pretraining_epochs(
                        raw, file_info['seizures']
                    )
                    labels = seizure_mask.astype(int)
                    metadata = [{'time': t, 'has_seizure': s} 
                            for t, s in zip(times, seizure_mask)]
                
                if epochs is not None and len(epochs) > 0:
                    # Normalize
                    epochs = self.normalize_epochs(epochs)
                    
                    # Detect artifacts
                    bad_mask = self.detect_artifacts(epochs)
                    
                    # Keep good epochs
                    good_idx = ~bad_mask
                    if np.sum(good_idx) > 0:
                        all_epochs.append(epochs[good_idx])
                        all_labels.append(labels[good_idx])
                        all_metadata.append([m for m, g in zip(metadata, good_idx) if g])
                    
            except Exception as e:
                if verbose:
                    print(f"    Warning: Error processing {file_info['filename']}: {e}")
                continue
        
        if len(all_epochs) == 0:
            return None
        
        # Handle variable channel counts 
        from collections import Counter
        channel_counts = [e.shape[1] for e in all_epochs]
        count_freq = Counter(channel_counts)
        most_common_n_channels = count_freq.most_common(1)[0][0]
        
        if verbose and len(count_freq) > 1:
            print(f"  Variable channels detected: {dict(count_freq)}")
            print(f"  Keeping only {most_common_n_channels}-channel epochs")
        
        # Filter epochs with consistent channel count
        filtered_epochs = []
        filtered_labels = []
        filtered_metadata = []
        
        for epochs, labels, meta in zip(all_epochs, all_labels, all_metadata):
            if epochs.shape[1] == most_common_n_channels:
                filtered_epochs.append(epochs)
                filtered_labels.append(labels)
                filtered_metadata.extend(meta)
        
        if len(filtered_epochs) == 0:
            return None
        
        # Safe to concatenate now
        epochs = np.concatenate(filtered_epochs, axis=0)
        labels = np.concatenate(filtered_labels, axis=0)
        
        result = {
            'subject_id': subject_id,
            'epochs': epochs,
            'labels': labels,
            'metadata': filtered_metadata,
            'sfreq': self.config.SFREQ_TARGET,
            'n_channels': most_common_n_channels,
            'epoch_duration': self.config.EPOCH_DURATION,
            'mode': mode
        }
        
        if verbose:
            n_seizure = np.sum(labels == 1)
            n_normal = np.sum(labels == 0)
            print(f"  ✓ {subject_id}: {len(epochs)} epochs "
                f"({n_seizure} seizure, {n_normal} normal), "
                f"{most_common_n_channels} channels")
        
        return result

In [7]:
# BATCH PROCESSING AND SAVING

# Create config instance and preprocessor
config = CHBMITConfig()
preprocessor = CHBMITPreprocessor(config)

def process_all_subjects(subjects, mode='pretraining', save_individual=True, 
                         save_combined=False, verbose=True):  # Added save_combined flag
    """Process subjects with checkpointing - can resume if interrupted."""
    
    all_data = []
    
    for subject_id in tqdm(subjects, desc=f"Processing ({mode})"):
        output_file = config.PROCESSED_DIR / f"chbmit_{subject_id}_{mode}.h5"
        
        # Skip if already processed (checkpoint)
        if output_file.exists():
            if verbose:
                print(f"  Skipping {subject_id} - already processed")
            # Only load if we need combined file
            if save_combined:
                try:
                    existing = load_from_hdf5(output_file)
                    all_data.append(existing)
                except:
                    os.remove(output_file)
                    print(f"  ⚠ Removed corrupted file for {subject_id}, will reprocess")
            continue
        
        try:
            result = preprocessor.process_subject(subject_id, mode=mode, verbose=verbose)
            
            if result is not None:
                if save_combined:
                    all_data.append(result)
                
                if save_individual:
                    save_to_hdf5(result, output_file)
                    
        except Exception as e:
            print(f"  ✗ Error processing {subject_id}: {e}")
            continue
    
    # Save combined dataset only if requested
    if save_combined and all_data:
        combined = combine_subjects(all_data)
        combined_file = config.PROCESSED_DIR / f"chbmit_combined_{mode}.h5"
        save_to_hdf5(combined, combined_file)
        return combined
    
    return None

def combine_subjects(subjects_data):
    """Combine data from multiple subjects."""
    all_epochs = []
    all_labels = []
    all_subject_ids = []
    
    # Find the most common channel count
    from collections import Counter
    channel_counts = [d['n_channels'] for d in subjects_data]
    most_common_channels = Counter(channel_counts).most_common(1)[0][0]
    
    for data in subjects_data:
        # Only include subjects with matching channel count
        if data['n_channels'] == most_common_channels:
            all_epochs.append(data['epochs'])
            all_labels.append(data['labels'])
            subj_id = data.get('subject_id', 'unknown')
            all_subject_ids.extend([subj_id] * len(data['epochs']))
    
    return {
        'epochs': np.concatenate(all_epochs, axis=0),
        'labels': np.concatenate(all_labels, axis=0),
        'subject_ids': np.array(all_subject_ids),
        'sfreq': subjects_data[0]['sfreq'],
        'n_channels': most_common_channels,
        'epoch_duration': subjects_data[0]['epoch_duration'],
        'mode': subjects_data[0]['mode']
    }


def save_to_hdf5(data, output_path):
    """Save processed data to HDF5 file with atomic write."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    temp_path = output_path.with_suffix('.h5.tmp')
    
    try:
        with h5py.File(temp_path, 'w') as f:
            epochs = data['epochs']
            chunk_size = min(100, len(epochs))
            
            f.create_dataset('epochs', data=epochs, 
                           compression='gzip', compression_opts=4,
                           chunks=(chunk_size, epochs.shape[1], epochs.shape[2]))
            f.create_dataset('labels', data=data['labels'], compression='gzip')
            
            # Handle both individual files (subject_id) and combined files (subject_ids)
            if 'subject_ids' in data:
                subject_ids = np.array(data['subject_ids'], dtype='S10')
                f.create_dataset('subject_ids', data=subject_ids)
            
            f.attrs['sfreq'] = data['sfreq']
            f.attrs['n_channels'] = data['n_channels']
            f.attrs['epoch_duration'] = data['epoch_duration']
            f.attrs['mode'] = data['mode']
            f.attrs['dataset'] = 'chb-mit'
            
            # Save subject_id for individual files
            if 'subject_id' in data:
                f.attrs['subject_id'] = data['subject_id']
            
            f.flush()
        
        if temp_path.exists():
            if output_path.exists():
                os.remove(output_path)
            temp_path.rename(output_path)
            print(f"  ✓ Saved: {output_path.name}")
            
    except Exception as e:
        if temp_path.exists():
            os.remove(temp_path)
        raise e


def load_from_hdf5(input_path):
    """Load processed data from HDF5 file."""
    with h5py.File(input_path, 'r') as f:
        data = {
            'epochs': f['epochs'][:],
            'labels': f['labels'][:],
            'sfreq': f.attrs['sfreq'],
            'n_channels': f.attrs['n_channels'],
            'epoch_duration': f.attrs['epoch_duration'],
            'mode': f.attrs['mode']
        }
        
        if 'subject_ids' in f:
            data['subject_ids'] = np.array([s.decode() for s in f['subject_ids'][:]])
        
        # Load subject_id for individual files
        if 'subject_id' in f.attrs:
            data['subject_id'] = f.attrs['subject_id']
    
    return data

print("✓ Processing functions defined")
print(f"  Config PROCESSED_DIR: {config.PROCESSED_DIR}")

✓ Processing functions defined
  Config PROCESSED_DIR: data/processed


In [8]:
# Clean up any corrupted files
processed_dir = Path('data/processed')

# Remove temp files
for f in processed_dir.glob('*.h5.tmp'):
    os.remove(f)
    print(f"Removed temp: {f}")

# Check for corrupted h5 files
for f in processed_dir.glob('chbmit_*.h5'):
    try:
        with h5py.File(f, 'r') as hf:
            _ = hf['epochs'].shape
        print(f"✓ Valid: {f.name}")
    except Exception as e:
        os.remove(f)
        print(f"✗ Removed corrupted: {f.name}")

✓ Valid: chbmit_chb01_pretraining.h5
✓ Valid: chbmit_chb01_seizure_detection.h5
✓ Valid: chbmit_chb03_pretraining.h5
✓ Valid: chbmit_chb03_seizure_detection.h5
✓ Valid: chbmit_chb05_pretraining.h5
✓ Valid: chbmit_chb05_seizure_detection.h5
✓ Valid: chbmit_chb08_pretraining.h5
✓ Valid: chbmit_chb10_pretraining.h5
✓ Valid: chbmit_chb12_pretraining.h5
✓ Valid: chbmit_chb14_pretraining.h5
✓ Valid: chbmit_chb17_pretraining.h5
✓ Valid: chbmit_chb19_pretraining.h5
✓ Valid: chbmit_chb20_pretraining.h5
✓ Valid: chbmit_chb22_pretraining.h5
✓ Valid: chbmit_combined_seizure_detection.h5


In [9]:
# Check disk space

import shutil

total, used, free = shutil.disk_usage("/")
print(f"Disk space:")
print(f"  Total: {total // (1024**3)} GB")
print(f"  Used:  {used // (1024**3)} GB")
print(f"  Free:  {free // (1024**3)} GB")

Disk space:
  Total: 199 GB
  Used:  131 GB
  Free:  68 GB


In [10]:
# Check what individual files are already saved
processed_dir = Path('data/processed')

print("Individual files saved:")
print("=" * 50)

total_epochs_seizure = 0
total_epochs_pretrain = 0

for f in sorted(processed_dir.glob('chbmit_chb*_*.h5')):
    try:
        with h5py.File(f, 'r') as hf:
            n_epochs = hf['epochs'].shape[0]
            mode = hf.attrs['mode']
            size_mb = f.stat().st_size / (1024**2)
            print(f"  ✓ {f.name}: {n_epochs:,} epochs ({size_mb:.1f} MB)")
            
            if 'seizure' in mode:
                total_epochs_seizure += n_epochs
            else:
                total_epochs_pretrain += n_epochs
    except Exception as e:
        print(f"  ✗ {f.name}: CORRUPTED - {e}")

print(f"\nTotal seizure detection epochs: {total_epochs_seizure:,}")
print(f"Total pretraining epochs: {total_epochs_pretrain:,}")

Individual files saved:
  ✓ chbmit_chb01_pretraining.h5: 4,865 epochs (2462.4 MB)
  ✓ chbmit_chb01_seizure_detection.h5: 96 epochs (48.7 MB)
  ✓ chbmit_chb03_pretraining.h5: 4,560 epochs (2308.5 MB)
  ✓ chbmit_chb03_seizure_detection.h5: 88 epochs (44.7 MB)
  ✓ chbmit_chb05_pretraining.h5: 4,680 epochs (2369.0 MB)
  ✓ chbmit_chb05_seizure_detection.h5: 136 epochs (69.0 MB)
  ✓ chbmit_chb08_pretraining.h5: 2,400 epochs (1215.1 MB)
  ✓ chbmit_chb10_pretraining.h5: 6,000 epochs (3037.2 MB)
  ✓ chbmit_chb12_pretraining.h5: 1,381 epochs (881.8 MB)
  ✓ chbmit_chb14_pretraining.h5: 3,120 epochs (1922.9 MB)
  ✓ chbmit_chb17_pretraining.h5: 2,400 epochs (1473.0 MB)
  ✓ chbmit_chb19_pretraining.h5: 3,471 epochs (2139.0 MB)
  ✓ chbmit_chb20_pretraining.h5: 3,309 epochs (2039.1 MB)
  ✓ chbmit_chb22_pretraining.h5: 3,720 epochs (1883.8 MB)

Total seizure detection epochs: 320
Total pretraining epochs: 39,906


In [11]:
# Remove failed temp files
for f in processed_dir.glob('*.h5.tmp'):
    os.remove(f)
    print(f"Removed: {f}")

In [12]:
# MAIN EXECUTION
print("=" * 60)
print("CHB-MIT SEIZURE DATASET PREPROCESSING")
print("=" * 60)

# Step 1: Find available subjects
chb_dir = CHBMITConfig.RAW_DIR

available_subjects = []
if chb_dir.exists():
    available_subjects = sorted([
        d.name for d in chb_dir.iterdir() 
        if d.is_dir() and d.name.startswith("chb")
    ])

print(f"\nFound {len(available_subjects)} subjects: {available_subjects}")

if len(available_subjects) == 0:
    print("\n No data found. Please download the dataset first.")
    print(f"   Expected location: {chb_dir.absolute()}")
    
else:
    # Step 2: Process for seizure detection (individual files only)
    print("\n" + "=" * 60)
    print("PROCESSING FOR SEIZURE DETECTION")
    print("=" * 60)
    process_all_subjects(
        subjects=available_subjects,
        mode='seizure_detection',
        save_individual=True,
        save_combined=False,  # Skip combined to avoid disk space issues
        verbose=True
    )
    
    # Step 3: Process for pretraining (individual files only)
    print("\n" + "=" * 60)
    print("PROCESSING FOR PRETRAINING")
    print("=" * 60)
    process_all_subjects(
        subjects=available_subjects,
        mode='pretraining',
        save_individual=True,
        save_combined=False,  # Skip combined to avoid disk space issues
        verbose=True
    )
    
    print("\n" + "=" * 60)
    print("✓ CHB-MIT PREPROCESSING COMPLETE!")
    print("=" * 60)

# Verify saved files
print("\n" + "=" * 60)
print("SAVED FILES")
print("=" * 60)

saved_files = list(CHBMITConfig.PROCESSED_DIR.glob("chbmit_*.h5"))
total_size = 0

for f in sorted(saved_files):
    size_mb = f.stat().st_size / (1024 * 1024)
    total_size += size_mb
    print(f"  ✓ {f.name} ({size_mb:.1f} MB)")

print(f"\nTotal: {len(saved_files)} files, {total_size:.1f} MB")
print(f"Location: {CHBMITConfig.PROCESSED_DIR.absolute()}")

CHB-MIT SEIZURE DATASET PREPROCESSING

Found 12 subjects: ['chb01', 'chb03', 'chb05', 'chb08', 'chb10', 'chb12', 'chb14', 'chb15', 'chb17', 'chb19', 'chb20', 'chb22']

PROCESSING FOR SEIZURE DETECTION
Processing (seizure_detection):   0%|          | 0/12 [00:00<?, ?it/s]  Skipping chb01 - already processed
  Skipping chb03 - already processed
  Skipping chb05 - already processed

Processing chb08...

  chb08 files:   0%|          | 0/20 [00:00<?, ?it/s]NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

  chb08 files:   5%|▌         | 1/20 [00:03<01:15,  3.99s/it]NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

  chb08 files:  10%|█         | 2/20 [00:06<00:53,  2.95s/it]NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

  chb08 files:  15%|█▌        | 3/20 [00:08<00:45,  2.70s/it]NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).

  chb08 files:  20%|██        | 4/20

In [13]:
# VERIFY FINAL OUTPUT
print("=" * 60)
print("VERIFICATION OF PROCESSED DATA")
print("=" * 60)

def verify_individual_files(processed_dir, mode):
    """Verify all individual subject files for a mode."""
    print(f"\n Mode: {mode}")
    print("-" * 40)
    
    files = sorted(Path(processed_dir).glob(f'chbmit_chb*_{mode}.h5'))
    
    total_epochs = 0
    total_seizure = 0
    total_normal = 0
    valid_files = 0
    
    for f in files:
        try:
            with h5py.File(f, 'r') as hf:
                n_epochs = hf['epochs'].shape[0]
                labels = hf['labels'][:]
                n_seizure = np.sum(labels == 1)
                n_normal = np.sum(labels == 0)
                
                total_epochs += n_epochs
                total_seizure += n_seizure
                total_normal += n_normal
                valid_files += 1
                
                print(f"  ✓ {f.name}: {n_epochs:,} epochs ({n_seizure} seizure, {n_normal} normal)")
        except Exception as e:
            print(f"  ✗ {f.name}: ERROR - {e}")
    
    print(f"\n  Summary for {mode}:")
    print(f"    Valid files: {valid_files}")
    print(f"    Total epochs: {total_epochs:,}")
    print(f"    Seizure: {total_seizure:,}, Normal: {total_normal:,}")
    
    total_hours = (total_epochs * 30) / 3600  # 30-second epochs
    print(f"    Total duration: {total_hours:.1f} hours")
    
    return valid_files > 0

# Verify both modes
all_valid = True
all_valid &= verify_individual_files(config.PROCESSED_DIR, 'seizure_detection')
all_valid &= verify_individual_files(config.PROCESSED_DIR, 'pretraining')

if all_valid:
    print("\n" + "=" * 60)
    print("ALL VERIFICATIONS PASSED - CHB-MIT PREPROCESSING COMPLETE")
    print("=" * 60)
    print("\nUse CHBMITDataset class to load data for training:")
    print("  dataset = CHBMITDataset('data/processed', mode='pretraining')")

VERIFICATION OF PROCESSED DATA

 Mode: seizure_detection
----------------------------------------
  ✓ chbmit_chb01_seizure_detection.h5: 96 epochs (24 seizure, 72 normal)
  ✓ chbmit_chb03_seizure_detection.h5: 88 epochs (22 seizure, 66 normal)
  ✓ chbmit_chb05_seizure_detection.h5: 136 epochs (34 seizure, 102 normal)

  Summary for seizure_detection:
    Valid files: 3
    Total epochs: 320
    Seizure: 80, Normal: 240
    Total duration: 2.7 hours

 Mode: pretraining
----------------------------------------
  ✓ chbmit_chb01_pretraining.h5: 4,865 epochs (24 seizure, 4841 normal)
  ✓ chbmit_chb03_pretraining.h5: 4,560 epochs (18 seizure, 4542 normal)
  ✓ chbmit_chb05_pretraining.h5: 4,680 epochs (23 seizure, 4657 normal)
  ✓ chbmit_chb08_pretraining.h5: 2,400 epochs (0 seizure, 2400 normal)
  ✓ chbmit_chb10_pretraining.h5: 6,000 epochs (0 seizure, 6000 normal)
  ✓ chbmit_chb12_pretraining.h5: 1,381 epochs (0 seizure, 1381 normal)
  ✓ chbmit_chb14_pretraining.h5: 3,120 epochs (0 seizure,

In [14]:
# Utility for later use
class CHBMITDataset:
    """Load CHB-MIT data from individual subject files without combining."""
    
    def __init__(self, processed_dir, mode='pretraining'):
        self.processed_dir = Path(processed_dir)
        self.mode = mode
        self.files = sorted(self.processed_dir.glob(f'chbmit_chb*_{mode}.h5'))
        
        # Build index mapping
        self.file_indices = []  # (file_idx, local_idx) for each global idx
        self.total_epochs = 0
        
        for file_idx, f in enumerate(self.files):
            with h5py.File(f, 'r') as hf:
                n_epochs = hf['epochs'].shape[0]
                for local_idx in range(n_epochs):
                    self.file_indices.append((file_idx, local_idx))
                self.total_epochs += n_epochs
        
        print(f"Loaded {len(self.files)} files, {self.total_epochs:,} total epochs")
    
    def __len__(self):
        return self.total_epochs
    
    def __getitem__(self, idx):
        file_idx, local_idx = self.file_indices[idx]
        with h5py.File(self.files[file_idx], 'r') as hf:
            epoch = hf['epochs'][local_idx]
            label = hf['labels'][local_idx]
        return epoch, label

# Test it
dataset = CHBMITDataset('data/processed', mode='pretraining')
print(f"\nDataset ready: {len(dataset):,} epochs")

Loaded 11 files, 39,906 total epochs

Dataset ready: 39,906 epochs


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>